In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from simulation_function import *
from simulation_function_euler import *

In [ ]:
'''
## param格式:
param = {
    "m1": 1.0,  # 摆球1质量 (kg)
    "m2": 1.0,  # 摆球2质量 (kg)
    "l1": 1.0,  # 摆绳1长度 (m)
    "l2": 1.0,  # 摆绳2长度 (m)
    "g": 9.81,  # 重力加速度 (m/s^2)
    "dt": 0.01,  # 时间步长 (s)
    "duration": 25.0,  # 总模拟时间 (s)
    "angle_mode": "DEG",  # 模式选择: 'DEG' (角度) 或 'RAD' (弧度)
    "theta1_0": 90.0,  # 初始角度1
    "theta2_0": 90.0002,  # 初始角度2
    "w1_0": 0.0,  # 初始角速度1 (无论模式，建议设为0)
    "w2_0": 0.0,  # 初始角速度2
}

"""

simulation function: 
simulate_double_pendulum(para = params)

"""
'''

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
param = {
    "m1": 1.0,  # 摆球1质量 (kg)
    "m2": 1.0,  # 摆球2质量 (kg)
    "l1": 1.0,  # 摆绳1长度 (m)
    "l2": 1.0,  # 摆绳2长度 (m)
    "g": 9.81,  # 重力加速度 (m/s^2)
    "dt": 0.01,  # 时间步长 (s)
    "duration": 300.0,  # 总模拟时间 (s)
    "angle_mode": "DEG",  # 模式选择: 'DEG' (角度) 或 'RAD' (弧度)
    "theta1_0": 90.0,  # 初始角度1
    "theta2_0": 90.0,  # 初始角度2
    "w1_0": 0.0,  # 初始角速度1 (无论模式，建议设为0)
    "w2_0": 0.0,  # 初始角速度2
}

### 实验B_01：靠近90°附近的初始角度微小差异对混沌系统的影响
**角度组**: 80°, 85°, 87.5°, 88.75°, 90°

每组两条线:
- **baseline**: theta1 = theta2 = angle
- **micro**: theta1 = angle, theta2 = angle + 0.01（差0.01度）

算法：RK4，时长300s

In [ ]:
import tqdm
import csv
from pathlib import Path

In [46]:
# ===== 基础配置 =====
run_number = "B_02"
folder_path = Path(f"run{run_number}")
folder_path.mkdir(exist_ok=True)

# 角度组
angle_groups = [80, 85, 87.5, 88.75, 90]

# 微扰量：0.01 度
epsilon = 0.01

# 固定步长，只用RK4
param["dt"] = 0.01
duration_str = f"{int(param['duration'])}s"

# 总实验数 = 5组 × 2条线 = 10
total_experiments = len(angle_groups) * 2

# ===== 开始循环 =====
num_counter = 1

with tqdm.tqdm(total=total_experiments, desc="Running Angle Sensitivity (RK4)") as pbar:
    for angle in angle_groups:
        # ---- 第1条线：基线 (baseline) ----
        param["theta1_0"] = float(angle)
        param["theta2_0"] = float(angle)
        
        num_str = f"{num_counter:03d}"
        # 用 angle 直接格式化，小数保留两位
        angle_str = f"{angle:.2f}".replace(".", "p")  # 87.50 → "87p50"
        filename = f"run_{run_number}_angle_{angle_str}deg_{num_str}_baseline_RK4_{param['dt']:.4f}_{duration_str}.csv"
        path = folder_path / filename
        
        temp_storage = simulate_double_pendulum(param, energy=True)
        
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['theta_1', 'theta_2', 'energy'])
            writer.writerows(temp_storage)
        
        num_counter += 1
        pbar.update(1)
        
        # ---- 第2条线：微扰 (micro) ----
        param["theta1_0"] = float(angle)
        param["theta2_0"] = float(angle) + epsilon
        
        num_str = f"{num_counter:03d}"
        filename = f"run_{run_number}_angle_{angle_str}deg_{num_str}_micro_RK4_{param['dt']:.4f}_{duration_str}.csv"
        path = folder_path / filename
        
        temp_storage = simulate_double_pendulum(param, energy=True)
        
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['theta_1', 'theta_2', 'energy'])
            writer.writerows(temp_storage)
        
        num_counter += 1
        pbar.update(1)

print("\n🎉 所有角度组的基线/微扰对比实验数据已完美落地！")
print(f"📁 数据已保存至: {folder_path}")

Running Angle Sensitivity (RK4): 100%|██████████| 10/10 [00:30<00:00,  3.08s/it]


🎉 所有角度组的基线/微扰对比实验数据已完美落地！
📁 数据已保存至: runB_02
